# 03 — Stabilizer simulator: tableaux, Bell + GHZ generators, scaling

This is the Phase 3 demo. We build small stabilizer states with the tableau simulator (`qec.stabilizer.StabilizerState`), inspect their generators, cross-check the reconstructed state vector against `qec.statevec` on a random Clifford circuit, and finally time the simulator on `n = 50` qubits — well past the size where state-vector simulation is even feasible.

Read alongside `notes/05-stabilizer-formalism.md`.

---


In [ ]:
import time
import numpy as np

from qec import statevec as sv
from qec.pauli import Pauli
from qec.stabilizer import StabilizerState

ATOL = 1e-10


## 1. Initial state |0...0>

The initial stabilizers should be `Z_q` on each qubit, with no signs.


In [ ]:
st = StabilizerState(4)
print("stabilizers:", st.stabilizer_generators())
print("destabilizers:", st.destabilizer_generators())


## 2. Bell state stabilizers

`H q0; CNOT(0->1)` should produce a state stabilised by `XX` and `ZZ` (the canonical Bell stabilizers).


In [ ]:
st = StabilizerState(2)
st.h(0)
st.cnot(0, 1)
print("Bell stabilizers:    ", st.stabilizer_generators())
print("Bell destabilizers:  ", st.destabilizer_generators())

# Reconstruct the state vector from the tableau and compare.
psi_stab = st.to_statevector()
psi_ref  = (sv.basis_state(2, 0b00) + sv.basis_state(2, 0b11)) / np.sqrt(2)
overlap  = abs(np.vdot(psi_stab, psi_ref))
print(f"|<Bell_ref | Bell_tableau>| = {overlap:.12f}  (should be 1.000000000000)")
assert np.isclose(overlap, 1.0, atol=ATOL)
print("OK")


## 3. GHZ state and its 5 generators

For an `n`-qubit GHZ, one set of stabilizer generators is `XXX...X` together with `Z_i Z_{i+1}` for `i = 0..n-2`. Let's verify on `n = 5`.


In [ ]:
n = 5
st = StabilizerState(n)
st.h(0)
for q in range(1, n):
    st.cnot(0, q)
gens = st.stabilizer_generators()
for g in gens:
    print(g)

# The simulator may pick a different (but equivalent) generating set; what we
# can verify directly is that *all* of XX...X and Z_i Z_{i+1} are stabilizers
# (i.e. they fix the state vector).
psi = st.to_statevector()
candidate_strs = ["XXXXX"] + [
    "I" * i + "Z" + "Z" + "I" * (n - 2 - i) for i in range(n - 1)
]
for s in candidate_strs:
    P = Pauli.from_string(s)
    out = P.matrix() @ psi
    ok = np.allclose(out, psi, atol=ATOL)
    print(f"{s:8s}  stabilises GHZ? {ok}")
    assert ok
print("OK")


## 4. Random Clifford circuit, tableau vs state vector

The Phase 3 acceptance gate. Run a 60-gate random Clifford circuit on `n = 4` qubits, simulating it both ways, and confirm that the reconstructed state vector matches up to global phase.


In [ ]:
n, depth = 4, 60
rng = np.random.default_rng(2026)

st = StabilizerState(n)
psi = sv.zero_state(n)

for _ in range(depth):
    kind = rng.integers(0, 4)
    q = int(rng.integers(0, n))
    if kind == 0:
        st.h(q);     psi = sv.apply_1q(psi, sv.H, q)
    elif kind == 1:
        st.s(q);     psi = sv.apply_1q(psi, sv.S, q)
    elif kind == 2:
        st.x(q);     psi = sv.apply_1q(psi, sv.X, q)
    else:
        q2 = int(rng.integers(0, n))
        while q2 == q:
            q2 = int(rng.integers(0, n))
        st.cnot(q, q2)
        psi = sv.cnot(psi, q, q2)

psi_stab = st.to_statevector()
overlap  = abs(np.vdot(psi_stab, psi))
print(f"|<sv | tableau>| = {overlap:.12f}  (should be 1.000000000000)")
assert np.isclose(overlap, 1.0, atol=ATOL)
print("OK")


## 5. Scaling: 50 qubits, 50 000 Clifford gates

State-vector simulation at `n = 50` is impossible: 2^50 ~ 10^15 amplitudes ~ 16 PB at complex128. The tableau simulator handles it in a couple of seconds.

The point of this cell is to feel the polynomial scaling. We don't reconstruct the state vector (we couldn't), but we do print one stabilizer generator at the end as evidence that the simulation completed.


In [ ]:
n, depth = 50, 50_000
rng = np.random.default_rng(0)
st = StabilizerState(n)

t0 = time.perf_counter()
for _ in range(depth):
    kind = rng.integers(0, 3)
    q = int(rng.integers(0, n))
    if kind == 0:
        st.h(q)
    elif kind == 1:
        st.s(q)
    else:
        q2 = int(rng.integers(0, n))
        while q2 == q:
            q2 = int(rng.integers(0, n))
        st.cnot(q, q2)
elapsed = time.perf_counter() - t0
print(f"{depth} Clifford gates on {n} qubits in {elapsed:.2f} s")
print("first stabilizer generator after evolution:")
print(" ", st.stabilizer_generators()[0])


## 6. Measurement statistics on |+>

Measuring `H|0> = |+>` in the Z basis gives 0/1 with equal probability. We sample 4000 shots and check.


In [ ]:
rng = np.random.default_rng(7)
template = StabilizerState(1)
template.h(0)

shots = 4000
ones = 0
for _ in range(shots):
    st = template.copy()
    ones += st.measure(0, rng=rng)
print(f"P(1) over {shots} shots: {ones/shots:.4f}  (expected ~ 0.5)")


## What this notebook gates

- The tableau simulator's stabilizer generators agree with textbook stabilizers for Bell, GHZ.
- Reconstructed state vectors match `qec.statevec` to 1e-10 on a 60-gate random Clifford circuit.
- 50k Clifford gates on 50 qubits run in seconds — the polynomial scaling promised by Gottesman-Knill.

Next: Phase 4 — the repetition code. We'll use the simulator above to encode, inject errors, extract syndromes, and plot the logical-vs-physical error-rate curve.
